# sensitive.ipynb

Speech-perturbation sensitivity analysis, including probe shifts, intensity, delivery cues, and cosine distances.

Point the model paths to the saved probes and baseline/perturbation outputs, then run the selected analysis sections.

Use the setup steps in the repository README before running this notebook.


In [ ]:
import os
import re
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {
    "QWEN": {
        "base_dir": "/content/drive/MyDrive/probing/full_probe/albe",
        "probe_dir": "/content/drive/MyDrive/probe/probe_representations_2",
    },

    "FLAMINGO": {
        "base_dir": "/content/drive/MyDrive/probing/full_probe_flamingo/abla",
        "probe_dir": "/content/drive/MyDrive/probe/probe_representations_flamingo_2",
    }
}

IDX2EMO = {
    0: "ANG",
    1: "DIS",
    2: "FEA",
    3: "HAP",
    4: "SAD",
    5: "NEU",
    6: "SUR"
}

EMO2IDX = {
    v: k for k, v in IDX2EMO.items()
}

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(self, input_dim, num_classes=7):
        super().__init__()

        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# =========================================================
# HELPERS
# =========================================================

def natural_sort_key(layer):

    nums = re.findall(r"\d+", layer)

    if nums:
        return (
            layer.split(nums[-1])[0],
            int(nums[-1])
        )

    return (layer, 0)


def get_all_probe_layers(probe_dir):

    probe_files = [
        f for f in os.listdir(probe_dir)
        if f.startswith("probe_") and f.endswith(".pt")
    ]

    layers = [
        f.replace("probe_", "").replace(".pt", "")
        for f in probe_files
    ]

    return sorted(
        layers,
        key=natural_sort_key
    )


def load_probes(probe_dir, layers):

    probes = {}

    for layer in layers:

        path = os.path.join(
            probe_dir,
            f"probe_{layer}.pt"
        )

        if not os.path.exists(path):
            continue

        ckpt = torch.load(
            path,
            map_location="cpu"
        )

        model = LinearProbe(
            ckpt["input_dim"]
        )

        model.load_state_dict(
            ckpt["state_dict"]
        )

        model.to(DEVICE)
        model.eval()

        probes[layer] = model

    print(f"Loaded {len(probes)} probes")

    return probes


def preprocess_rep(x, layer):

    x = x.float()

    # =====================================================
    # LM pooling
    # =====================================================

    if layer.startswith("lm_") and x.dim() == 3:
        x = x.mean(dim=1)

    # =====================================================
    # Shape handling
    # =====================================================

    if x.dim() == 2 and x.shape[0] == 1:
        x = x.squeeze(0)

    elif x.dim() == 2:
        x = x.mean(dim=0)

    elif x.dim() > 2:
        x = x.flatten()

    # =====================================================
    # Normalize
    # =====================================================

    x = (
        x - x.mean()
    ) / (x.std() + 1e-6)

    x = x.unsqueeze(0).to(DEVICE)

    return x


def compute_entropy(prob_vec):

    p = np.clip(prob_vec, 1e-12, 1.0)

    return -(p * np.log(p)).sum()


def extract_emotion_from_id(sample_id):

    """
    Example:
    1073_WSI_NEU_XX

    -> NEU
    """

    try:
        return str(sample_id).split("_")[2]

    except:
        return None

# =========================================================
# SHIFT ANALYSIS
# =========================================================

def run_shift_analysis_from_merged(
    merged_csv,
    probes
):

    df = pd.read_csv(merged_csv)

    print("\nDetected task types:")
    print(df["task"].value_counts())

    all_results = []

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):

        manip_rep_path = row["rep_path"]
        neutral_rep_path = row["neutral_rep_path"]

        # =================================================
        # TARGET EMOTION FROM ID
        # =================================================

        target_emotion = extract_emotion_from_id(
            row["id"]
        )

        if target_emotion not in EMO2IDX:
            continue

        target_idx = EMO2IDX[
            target_emotion
        ]

        # =================================================
        # LOAD REPRESENTATIONS
        # =================================================

        try:

            manip_rep = torch.load(
                manip_rep_path,
                map_location="cpu"
            )

            neutral_rep = torch.load(
                neutral_rep_path,
                map_location="cpu"
            )

        except Exception as e:

            print(f"Representation load error: {e}")
            continue

        # =================================================
        # PROBE EACH LAYER
        # =================================================

        for layer, model in probes.items():

            if layer not in manip_rep:
                continue

            if layer not in neutral_rep:
                continue

            try:

                # =============================================
                # MANIPULATED
                # =============================================

                x_manip = preprocess_rep(
                    manip_rep[layer],
                    layer
                )

                with torch.no_grad():

                    logits_manip = model(x_manip)

                    probs_manip = F.softmax(
                        logits_manip,
                        dim=-1
                    )

                    probs_manip = (
                        probs_manip.squeeze(0)
                        .cpu()
                        .numpy()
                    )

                pred_manip = IDX2EMO[
                    probs_manip.argmax()
                ]

                conf_manip = probs_manip.max()

                entropy_manip = compute_entropy(
                    probs_manip
                )

                # =============================================
                # NEUTRAL
                # =============================================

                x_neutral = preprocess_rep(
                    neutral_rep[layer],
                    layer
                )

                with torch.no_grad():

                    logits_neutral = model(
                        x_neutral
                    )

                    probs_neutral = F.softmax(
                        logits_neutral,
                        dim=-1
                    )

                    probs_neutral = (
                        probs_neutral.squeeze(0)
                        .cpu()
                        .numpy()
                    )

                pred_neutral = IDX2EMO[
                    probs_neutral.argmax()
                ]

                conf_neutral = probs_neutral.max()

                entropy_neutral = compute_entropy(
                    probs_neutral
                )

                # =============================================
                # TARGET EMOTION CONFIDENCE
                # =============================================

                target_conf_manip = probs_manip[
                    target_idx
                ]

                target_conf_neutral = probs_neutral[
                    target_idx
                ]

                target_conf_shift = (
                    target_conf_manip
                    - target_conf_neutral
                )

                # =============================================
                # SHIFTS
                # =============================================

                conf_shift = (
                    conf_manip
                    - conf_neutral
                )

                entropy_shift = (
                    entropy_manip
                    - entropy_neutral
                )

                # =============================================
                # EMOTION SHIFT RATE
                # =============================================

                acc_shift = int(
                    pred_manip != pred_neutral
                )

                # =============================================
                # SAVE
                # =============================================

                all_results.append({

                    "id": row["id"],
                    "task": row["task"],
                    "layer": layer,

                    # -----------------------------------------
                    # TARGET EMOTION
                    # -----------------------------------------

                    "target_emotion": target_emotion,

                    "target_conf_manip": target_conf_manip,
                    "target_conf_neutral": target_conf_neutral,

                    "target_conf_shift": target_conf_shift,

                    # -----------------------------------------
                    # ORIGINAL
                    # -----------------------------------------

                    "conf_shift": conf_shift,
                    "entropy_shift": entropy_shift,
                    "acc_shift": acc_shift,

                    "conf_manip": conf_manip,
                    "conf_neutral": conf_neutral,

                    "entropy_manip": entropy_manip,
                    "entropy_neutral": entropy_neutral,

                    "pred_manip": pred_manip,
                    "pred_neutral": pred_neutral,

                })

            except Exception as e:

                print(f"Error: {layer} | {e}")
                continue

    return pd.DataFrame(all_results)

# =========================================================
# VISUALIZATION
# =========================================================

def plot_shift_results(all_shift_df):

    fig, axes = plt.subplots(
        2,
        8,
        figsize=(42, 10)
    )

    metrics = [

        (
            "target_conf_shift",
            "Target Emotion Confidence Shift"
        ),

        (
            "conf_shift",
            "Max Confidence Shift"
        ),

        (
            "entropy_shift",
            "Entropy Shift"
        ),

        (
            "acc_shift",
            "Emotion Shift Rate"
        )
    ]

    for row_idx, (
        model_name,
        shift_df
    ) in enumerate(all_shift_df.items()):

        # =================================================
        # TASK TYPES
        # =================================================

        task_types = sorted(
            shift_df["task"].unique()
        )

        print(f"\n{model_name} task types:")
        print(task_types)

        # =================================================
        # SPLIT AUDIO / LLM
        # =================================================

        audio_df = shift_df[
            ~shift_df["layer"].str.startswith("lm_")
        ]

        llm_df = shift_df[
            shift_df["layer"].str.startswith("lm_")
        ]

        groups = [
            ("Audio + Projector", audio_df),
            ("LLM", llm_df)
        ]

        # =================================================
        # PLOT
        # =================================================

        for group_idx, (
            group_name,
            group_df
        ) in enumerate(groups):

            for metric_idx, (
                metric,
                title
            ) in enumerate(metrics):

                col_idx = (
                    group_idx * 4
                    + metric_idx
                )

                ax = axes[row_idx, col_idx]

                # ---------------------------------------------
                # PLOT EACH TASK TYPE
                # ---------------------------------------------

                colors = ["gold", "blue"]

                for task_i, task_name in enumerate(task_types):

                    task_subset = group_df[
                        group_df["task"] == task_name
                    ]

                    stats = task_subset.groupby(
                        "layer"
                    )[metric].agg(
                        ["mean", "std"]
                    ).reset_index()

                    stats["sort_key"] = stats[
                        "layer"
                    ].apply(natural_sort_key)

                    stats = stats.sort_values(
                        "sort_key"
                    )

                    x = np.arange(len(stats))

                    mean = stats["mean"].values
                    std = stats["std"].values

                    color = colors[
                        task_i % len(colors)
                    ]

                    ax.plot(
                        x,
                        mean,
                        label=task_name,
                        color=color
                    )

                    ax.fill_between(
                        x,
                        mean - std,
                        mean + std,
                        alpha=0.20,
                        color=color
                    )

                # ---------------------------------------------
                # STYLE
                # ---------------------------------------------

                ax.axhline(
                    0,
                    linestyle="--",
                    linewidth=1,
                    color="black"
                )

                ax.set_title(
                    f"{model_name}\n"
                    f"{group_name}\n"
                    f"{title}"
                )

                ax.set_xlabel("Layer")

                if col_idx == 0:
                    ax.set_ylabel("Shift")

                ax.grid(True, alpha=0.3)

                ax.legend()

    plt.tight_layout()
    plt.show()

# =========================================================
# MAIN
# =========================================================

all_shift_df = {}

for model_name, cfg in MODELS.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    base_dir = cfg["base_dir"]

    probe_dir = cfg["probe_dir"]

    merged_csv = os.path.join(
        base_dir,
        "merged_results.csv"
    )

    # =====================================================
    # LOAD PROBES
    # =====================================================

    layers = get_all_probe_layers(
        probe_dir
    )

    probes = load_probes(
        probe_dir,
        layers
    )

    # =====================================================
    # RUN SHIFT ANALYSIS
    # =====================================================

    shift_df = run_shift_analysis_from_merged(
        merged_csv,
        probes
    )

    # =====================================================
    # SAVE
    # =====================================================

    save_path = os.path.join(
        base_dir,
        "shift_results.csv"
    )

    shift_df.to_csv(
        save_path,
        index=False
    )

    print(f"\nSaved:")
    print(save_path)

    all_shift_df[model_name] = shift_df

# =========================================================
# FINAL PLOT
# =========================================================

plot_shift_results(all_shift_df)

In [ ]:
import os
import re
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {
    "QWEN": {
        "base_dir": "/content/drive/MyDrive/probing/full_probe/albe",
        "probe_dir": "/content/drive/MyDrive/probe/probe_representations_2",
    },

    "FLAMINGO": {
        "base_dir": "/content/drive/MyDrive/probing/full_probe_flamingo/abla",
        "probe_dir": "/content/drive/MyDrive/probe/probe_representations_flamingo_2",
    }
}

IDX2EMO = {
    0: "ANG",
    1: "DIS",
    2: "FEA",
    3: "HAP",
    4: "SAD",
    5: "NEU",
    6: "SUR"
}

EMO2IDX = {
    v: k for k, v in IDX2EMO.items()
}

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(self, input_dim, num_classes=7):
        super().__init__()

        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# =========================================================
# HELPERS
# =========================================================

def natural_sort_key(layer):

    nums = re.findall(r"\d+", layer)

    if nums:
        return (
            layer.split(nums[-1])[0],
            int(nums[-1])
        )

    return (layer, 0)

def get_all_probe_layers(probe_dir):

    probe_files = [
        f for f in os.listdir(probe_dir)
        if f.startswith("probe_") and f.endswith(".pt")
    ]

    layers = [
        f.replace("probe_", "").replace(".pt", "")
        for f in probe_files
    ]

    return sorted(
        layers,
        key=natural_sort_key
    )

def load_probes(probe_dir, layers):

    probes = {}

    for layer in layers:

        path = os.path.join(
            probe_dir,
            f"probe_{layer}.pt"
        )

        if not os.path.exists(path):
            continue

        ckpt = torch.load(
            path,
            map_location="cpu"
        )

        model = LinearProbe(
            ckpt["input_dim"]
        )

        model.load_state_dict(
            ckpt["state_dict"]
        )

        model.to(DEVICE)
        model.eval()

        probes[layer] = model

    print(f"Loaded {len(probes)} probes")

    return probes

def preprocess_rep(x, layer):

    x = x.float()

    # =====================================================
    # LM pooling
    # =====================================================

    if layer.startswith("lm_") and x.dim() == 3:
        x = x.mean(dim=1)

    # =====================================================
    # Shape handling
    # =====================================================

    if x.dim() == 2 and x.shape[0] == 1:
        x = x.squeeze(0)

    elif x.dim() == 2:
        x = x.mean(dim=0)

    elif x.dim() > 2:
        x = x.flatten()

    # =====================================================
    # Normalize
    # =====================================================

    x = (
        x - x.mean()
    ) / (x.std() + 1e-6)

    x = x.unsqueeze(0).to(DEVICE)

    return x

# =========================================================
# CONDITION EXTRACTION
# =========================================================

def extract_condition_from_path(path):

    """
    Example:
    subset_audio\\1073_WSI_NEU_XX_cough.wav

    -> cough
    """

    filename = os.path.basename(path)

    filename = filename.replace(".wav", "")

    parts = filename.split("_")

    condition = parts[-1]

    return condition

# =========================================================
# ANALYSIS
# =========================================================

def run_condition_accuracy_shift_analysis(
    merged_csv,
    probes
):

    df = pd.read_csv(merged_csv)

    # =====================================================
    # STYLE ONLY
    # =====================================================

    df = df[
        df["task"] == "style"
    ].copy()

    print("\nUsing STYLE task only")
    print(f"Samples: {len(df)}")

    all_results = []

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):

        manip_rep_path = row["rep_path"]

        neutral_rep_path = row["neutral_rep_path"]

        # =================================================
        # CONDITION
        # =================================================

        condition = extract_condition_from_path(
            row["manipulated_path"]
        )

        # =================================================
        # TARGET IS ALWAYS NEU
        # =================================================

        target_emotion = "NEU"

        # =================================================
        # LOAD REPRESENTATIONS
        # =================================================

        try:

            manip_rep = torch.load(
                manip_rep_path,
                map_location="cpu"
            )

            neutral_rep = torch.load(
                neutral_rep_path,
                map_location="cpu"
            )

        except Exception as e:

            print(f"Representation load error: {e}")
            continue

        # =================================================
        # PROBE EACH LAYER
        # =================================================

        for layer, model in probes.items():

            if layer not in manip_rep:
                continue

            if layer not in neutral_rep:
                continue

            try:

                # =============================================
                # MANIPULATED
                # =============================================

                x_manip = preprocess_rep(
                    manip_rep[layer],
                    layer
                )

                with torch.no_grad():

                    logits_manip = model(x_manip)

                    probs_manip = F.softmax(
                        logits_manip,
                        dim=-1
                    )

                    probs_manip = (
                        probs_manip.squeeze(0)
                        .cpu()
                        .numpy()
                    )

                pred_manip = IDX2EMO[
                    probs_manip.argmax()
                ]

                # =============================================
                # NEUTRAL
                # =============================================

                x_neutral = preprocess_rep(
                    neutral_rep[layer],
                    layer
                )

                with torch.no_grad():

                    logits_neutral = model(
                        x_neutral
                    )

                    probs_neutral = F.softmax(
                        logits_neutral,
                        dim=-1
                    )

                    probs_neutral = (
                        probs_neutral.squeeze(0)
                        .cpu()
                        .numpy()
                    )

                pred_neutral = IDX2EMO[
                    probs_neutral.argmax()
                ]

                # =============================================
                # ACCURACY
                # =============================================

                manip_correct = int(
                    pred_manip == target_emotion
                )

                neutral_correct = int(
                    pred_neutral == target_emotion
                )

                acc_shift = (
                    manip_correct
                    - neutral_correct
                )

                # =============================================
                # SAVE
                # =============================================

                all_results.append({

                    "id": row["id"],

                    "condition": condition,

                    "layer": layer,

                    "pred_manip": pred_manip,
                    "pred_neutral": pred_neutral,

                    "manip_correct": manip_correct,
                    "neutral_correct": neutral_correct,

                    "acc_shift": acc_shift

                })

            except Exception as e:

                print(f"Error: {layer} | {e}")
                continue

    return pd.DataFrame(all_results)

# =========================================================
# PLOT
# =========================================================

def plot_condition_accuracy_shift(
    all_results
):

    conditions = sorted(
        all_results["condition"].unique()
    )

    print("\nConditions:")
    print(conditions)

    # =====================================================
    # COLORS
    # =====================================================

    cmap = plt.cm.get_cmap(
        "tab10",
        len(conditions)
    )

    # =====================================================
    # FIGURE
    # =====================================================

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(20, 10)
    )

    axes = axes.flatten()

    for plot_idx, (
        model_name,
        shift_df
    ) in enumerate(all_results.items()):

        # =================================================
        # AUDIO
        # =================================================

        audio_df = shift_df[
            ~shift_df["layer"].str.startswith("lm_")
        ]

        # =================================================
        # LLM
        # =================================================

        llm_df = shift_df[
            shift_df["layer"].str.startswith("lm_")
        ]

        groups = [
            ("Audio + Projector", audio_df),
            ("LLM", llm_df)
        ]

        for group_idx, (
            group_name,
            group_df
        ) in enumerate(groups):

            ax = axes[
                plot_idx * 2 + group_idx
            ]

            # =============================================
            # EACH CONDITION
            # =============================================

            for cond_idx, condition in enumerate(conditions):

                subset = group_df[
                    group_df["condition"] == condition
                ]

                stats = subset.groupby(
                    "layer"
                )["acc_shift"].agg(
                    ["mean", "std"]
                ).reset_index()

                stats["sort_key"] = stats[
                    "layer"
                ].apply(natural_sort_key)

                stats = stats.sort_values(
                    "sort_key"
                )

                x = np.arange(len(stats))

                mean = stats["mean"].values

                std = stats["std"].values

                color = cmap(cond_idx)

                ax.plot(
                    x,
                    mean,
                    label=condition,
                    color=color,
                    linewidth=2
                )

                ax.fill_between(
                    x,
                    mean - std,
                    mean + std,
                    alpha=0.20,
                    color=color
                )

            # =============================================
            # STYLE
            # =============================================

            ax.axhline(
                0,
                linestyle="--",
                color="black",
                linewidth=1
            )

            ax.set_title(
                f"{model_name}\n{group_name}\nAccuracy Shift"
            )

            ax.set_xlabel("Layer")
            ax.set_ylabel("Accuracy Shift")

            ax.grid(True, alpha=0.3)

            ax.legend()

    plt.tight_layout()
    plt.show()

# =========================================================
# MAIN
# =========================================================

all_condition_results = {}

for model_name, cfg in MODELS.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    base_dir = cfg["base_dir"]

    probe_dir = cfg["probe_dir"]

    merged_csv = os.path.join(
        base_dir,
        "merged_results.csv"
    )

    # =====================================================
    # LOAD PROBES
    # =====================================================

    layers = get_all_probe_layers(
        probe_dir
    )

    probes = load_probes(
        probe_dir,
        layers
    )

    # =====================================================
    # RUN ANALYSIS
    # =====================================================

    condition_df = run_condition_accuracy_shift_analysis(
        merged_csv,
        probes
    )

    # =====================================================
    # SAVE
    # =====================================================

    save_path = os.path.join(
        base_dir,
        "condition_accuracy_shift.csv"
    )

    condition_df.to_csv(
        save_path,
        index=False
    )

    print(f"\nSaved:")
    print(save_path)

    all_condition_results[
        model_name
    ] = condition_df

# =========================================================
# FINAL PLOT
# =========================================================

plot_condition_accuracy_shift(
    all_condition_results
)

In [ ]:
def plot_condition_accuracy_shift(
    all_results
):
    # Get conditions from the first model's DataFrame
    first_model_df = list(all_results.values())[0]
    conditions = sorted(first_model_df["condition"].unique())

    print("\nConditions:")
    print(conditions)

    # =====================================================
    # COLORS
    # =====================================================

    cmap = plt.cm.get_cmap(
        "tab10",
        len(conditions)
    )

    # =====================================================
    # FIGURE
    # =====================================================

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(20, 10)
    )

    axes = axes.flatten()

    for plot_idx, (
        model_name,
        shift_df
    ) in enumerate(all_results.items()):

        # =================================================
        # AUDIO
        # =================================================

        audio_df = shift_df[
            ~shift_df["layer"].str.startswith("lm_")
        ]

        # =================================================
        # LLM
        # =================================================

        llm_df = shift_df[
            shift_df["layer"].str.startswith("lm_")
        ]

        groups = [
            ("Audio + Projector", audio_df),
            ("LLM", llm_df)
        ]

        for group_idx, (
            group_name,
            group_df
        ) in enumerate(groups):

            ax = axes[
                plot_idx * 2 + group_idx
            ]

            # Check if group_df is empty
            if group_df.empty:
                ax.text(0.5, 0.5, f"No {group_name} layers found",
                       ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f"{model_name}\n{group_name}\nAccuracy Shift")
                continue

            # =============================================
            # EACH CONDITION
            # =============================================

            for cond_idx, condition in enumerate(conditions):

                subset = group_df[
                    group_df["condition"] == condition
                ]

                if subset.empty:
                    continue

                stats = subset.groupby(
                    "layer"
                )["acc_shift"].agg(
                    ["mean", "std"]
                ).reset_index()

                stats["sort_key"] = stats[
                    "layer"
                ].apply(natural_sort_key)

                stats = stats.sort_values(
                    "sort_key"
                )

                x = np.arange(len(stats))

                mean = stats["mean"].values

                std = stats["std"].values

                color = cmap(cond_idx)

                ax.plot(
                    x,
                    mean,
                    label=condition,
                    color=color,
                    linewidth=2
                )

                ax.fill_between(
                    x,
                    mean - std,
                    mean + std,
                    alpha=0.20,
                    color=color
                )

            # =============================================
            # STYLE
            # =============================================

            ax.axhline(
                0,
                linestyle="--",
                color="black",
                linewidth=1
            )

            # Set x-ticks to layer names
            if not group_df.empty:
                layers = stats["layer"].tolist()
                ax.set_xticks(x)
                ax.set_xticklabels(layers, rotation=45, ha='right', fontsize=8)

            ax.set_title(
                f"{model_name}\n{group_name}\nAccuracy Shift"
            )

            ax.set_xlabel("Layer")
            ax.set_ylabel("Accuracy Shift")

            ax.grid(True, alpha=0.3)

            ax.legend(loc='best', fontsize=8)

    plt.tight_layout()
    plt.show()
# =========================================================
# FINAL PLOT
# =========================================================

plot_condition_accuracy_shift(
    all_condition_results
)

In [ ]:
import os
import re
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {

    "QWEN": {
        "base_dir":
        "/content/drive/MyDrive/probing/full_probe/albe",

        "probe_dir":
        "/content/drive/MyDrive/probe/probe_representations_2",
    },

    "FLAMINGO": {
        "base_dir":
        "/content/drive/MyDrive/probing/full_probe_flamingo/abla",

        "probe_dir":
        "/content/drive/MyDrive/probe/probe_representations_flamingo_2",
    }
}

IDX2EMO = {
    0: "ANG",
    1: "DIS",
    2: "FEA",
    3: "HAP",
    4: "SAD",
    5: "NEU",
    6: "SUR"
}

EMO2IDX = {
    v: k for k, v in IDX2EMO.items()
}

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(self, input_dim, num_classes=7):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            num_classes
        )

    def forward(self, x):

        return self.linear(x)

# =========================================================
# HELPERS
# =========================================================

def natural_sort_key(layer):

    nums = re.findall(r"\d+", layer)

    if nums:

        return (
            layer.split(nums[-1])[0],
            int(nums[-1])
        )

    return (layer, 0)


def get_all_probe_layers(probe_dir):

    probe_files = [

        f for f in os.listdir(probe_dir)

        if f.startswith("probe_")
        and f.endswith(".pt")
    ]

    layers = [

        f.replace("probe_", "")
         .replace(".pt", "")

        for f in probe_files
    ]

    return sorted(
        layers,
        key=natural_sort_key
    )


def load_probes(probe_dir, layers):

    probes = {}

    for layer in layers:

        path = os.path.join(
            probe_dir,
            f"probe_{layer}.pt"
        )

        if not os.path.exists(path):
            continue

        ckpt = torch.load(
            path,
            map_location="cpu"
        )

        model = LinearProbe(
            ckpt["input_dim"]
        )

        model.load_state_dict(
            ckpt["state_dict"]
        )

        model.to(DEVICE)
        model.eval()

        probes[layer] = model

    print(f"Loaded {len(probes)} probes")

    return probes


def preprocess_rep(x, layer):

    x = x.float()

    # =====================================================
    # LM pooling
    # =====================================================

    if layer.startswith("lm_") and x.dim() == 3:

        x = x.mean(dim=1)

    # =====================================================
    # Shape handling
    # =====================================================

    if x.dim() == 2 and x.shape[0] == 1:

        x = x.squeeze(0)

    elif x.dim() == 2:

        x = x.mean(dim=0)

    elif x.dim() > 2:

        x = x.flatten()

    # =====================================================
    # Normalize
    # =====================================================

    x = (
        x - x.mean()
    ) / (x.std() + 1e-6)

    x = x.unsqueeze(0).to(DEVICE)

    return x


def compute_entropy(prob_vec):

    p = np.clip(
        prob_vec,
        1e-12,
        1.0
    )

    return -(p * np.log(p)).sum()


def extract_emotion_from_id(sample_id):

    """
    STYLE:
    1073_WSI_NEU_XX
              ↑
             NEU

    INTENSITY:
    1023_IEO_ANG
             ↑
            ANG
    """

    try:

        sample_id = str(sample_id)

        sample_id = os.path.splitext(
            sample_id
        )[0]

        parts = sample_id.split("_")

        # style task
        if parts[-1] == "XX":

            return parts[-2].upper()

        # intensity task
        else:

            return parts[-1].upper()

    except:

        return None


def extract_intensity(path):

    """
    Example:
    1088_IEO_SAD_MD.wav

    -> MD
    """

    try:

        filename = str(path).split("\\")[-1]

        filename = os.path.splitext(
            filename
        )[0]

        parts = filename.split("_")

        intensity = parts[-1].upper()

        if intensity in ["LO", "MD", "HI"]:

            return intensity

        return None

    except:

        return None

# =========================================================
# SHIFT ANALYSIS
# =========================================================

def run_shift_analysis_from_merged(
    merged_csv,
    probes
):

    df = pd.read_csv(merged_csv)

    print("\nDetected task types:")
    print(df["task"].value_counts())

    # =====================================================
    # EXTRACT INTENSITY FROM manipulated_path
    # =====================================================

    df["intensity_level"] = df[
        "manipulated_path"
    ].apply(
        extract_intensity
    )

    print("\nIntensity distribution:")
    print(
        df["intensity_level"]
        .value_counts(dropna=False)
    )

    all_results = []

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):

        manip_rep_path = row["rep_path"]

        neutral_rep_path = row[
            "neutral_rep_path"
        ]

        # =================================================
        # TARGET EMOTION
        # =================================================

        target_emotion = extract_emotion_from_id(
            row["id"]
        )

        if target_emotion not in EMO2IDX:
            continue

        target_idx = EMO2IDX[
            target_emotion
        ]

        # =================================================
        # INTENSITY
        # =================================================

        intensity_level = row[
            "intensity_level"
        ]

        # =================================================
        # LOAD REPRESENTATIONS
        # =================================================

        try:

            manip_rep = torch.load(
                manip_rep_path,
                map_location="cpu"
            )

            neutral_rep = torch.load(
                neutral_rep_path,
                map_location="cpu"
            )

        except Exception as e:

            print(f"Representation load error: {e}")
            continue

        # =================================================
        # PROBE EACH LAYER
        # =================================================

        for layer, model in probes.items():

            if layer not in manip_rep:
                continue

            if layer not in neutral_rep:
                continue

            try:

                # =============================================
                # MANIPULATED
                # =============================================

                x_manip = preprocess_rep(
                    manip_rep[layer],
                    layer
                )

                with torch.no_grad():

                    logits_manip = model(
                        x_manip
                    )

                    probs_manip = F.softmax(
                        logits_manip,
                        dim=-1
                    )

                    probs_manip = (
                        probs_manip
                        .squeeze(0)
                        .cpu()
                        .numpy()
                    )

                pred_manip = IDX2EMO[
                    probs_manip.argmax()
                ]

                conf_manip = probs_manip.max()

                entropy_manip = compute_entropy(
                    probs_manip
                )

                # =============================================
                # NEUTRAL
                # =============================================

                x_neutral = preprocess_rep(
                    neutral_rep[layer],
                    layer
                )

                with torch.no_grad():

                    logits_neutral = model(
                        x_neutral
                    )

                    probs_neutral = F.softmax(
                        logits_neutral,
                        dim=-1
                    )

                    probs_neutral = (
                        probs_neutral
                        .squeeze(0)
                        .cpu()
                        .numpy()
                    )

                pred_neutral = IDX2EMO[
                    probs_neutral.argmax()
                ]

                conf_neutral = probs_neutral.max()

                entropy_neutral = compute_entropy(
                    probs_neutral
                )

                # =============================================
                # TARGET EMOTION CONFIDENCE
                # =============================================

                target_conf_manip = probs_manip[
                    target_idx
                ]

                target_conf_neutral = probs_neutral[
                    target_idx
                ]

                target_conf_shift = (
                    target_conf_manip
                    - target_conf_neutral
                )

                # =============================================
                # OTHER SHIFTS
                # =============================================

                conf_shift = (
                    conf_manip
                    - conf_neutral
                )

                entropy_shift = (
                    entropy_manip
                    - entropy_neutral
                )

                acc_shift = int(
                    pred_manip != pred_neutral
                )

                # =============================================
                # SAVE
                # =============================================

                all_results.append({

                    "id": row["id"],

                    "task": row["task"],

                    "intensity_level":
                    intensity_level,

                    "layer": layer,

                    "target_emotion":
                    target_emotion,

                    "target_conf_shift":
                    target_conf_shift,

                    "conf_shift":
                    conf_shift,

                    "entropy_shift":
                    entropy_shift,

                    "acc_shift":
                    acc_shift,

                    "pred_manip":
                    pred_manip,

                    "pred_neutral":
                    pred_neutral,
                })

            except Exception as e:

                print(f"Error: {layer} | {e}")
                continue

    return pd.DataFrame(all_results)

# =========================================================
# VISUALIZATION
# =========================================================

def plot_shift_results(all_shift_df):

    fig, axes = plt.subplots(
        2,
        8,
        figsize=(42, 10)
    )

    metrics = [

        (
            "target_conf_shift",
            "Target Emotion Confidence Shift"
        ),

        (
            "conf_shift",
            "Max Confidence Shift"
        ),

        (
            "entropy_shift",
            "Entropy Shift"
        ),

        (
            "acc_shift",
            "Emotion Shift Rate"
        )
    ]

    for row_idx, (
        model_name,
        shift_df
    ) in enumerate(all_shift_df.items()):

        # =================================================
        # ONLY INTENSITY TASK
        # =================================================

        intensity_df = shift_df[
            shift_df["task"] == "intensity"
        ]

        # =================================================
        # KEEP ONLY LO / HI
        # =================================================

        intensity_df = intensity_df[
            intensity_df["intensity_level"]
            .isin(["LO", "HI"])
        ]

        print(f"\n{model_name}")
        print(
            intensity_df[
                "intensity_level"
            ].value_counts()
        )

        # =================================================
        # SPLIT AUDIO / LLM
        # =================================================

        audio_df = intensity_df[
            ~intensity_df["layer"]
            .str.startswith("lm_")
        ]

        llm_df = intensity_df[
            intensity_df["layer"]
            .str.startswith("lm_")
        ]

        groups = [

            ("Audio + Projector", audio_df),

            ("LLM", llm_df)
        ]

        # =================================================
        # PLOT
        # =================================================

        for group_idx, (
            group_name,
            group_df
        ) in enumerate(groups):

            for metric_idx, (
                metric,
                title
            ) in enumerate(metrics):

                col_idx = (
                    group_idx * 4
                    + metric_idx
                )

                ax = axes[
                    row_idx,
                    col_idx
                ]

                # =============================================
                # LO / HI LINES
                # =============================================

                intensity_colors = {

                    "LO": "blue",
                    "HI": "red"
                }

                for intensity in ["LO", "HI"]:

                    subset = group_df[
                        group_df[
                            "intensity_level"
                        ] == intensity
                    ]

                    if len(subset) == 0:
                        continue

                    stats = subset.groupby(
                        "layer"
                    )[metric].agg(
                        ["mean", "std"]
                    ).reset_index()

                    stats["sort_key"] = stats[
                        "layer"
                    ].apply(
                        natural_sort_key
                    )

                    stats = stats.sort_values(
                        "sort_key"
                    )

                    x = np.arange(len(stats))

                    mean = stats[
                        "mean"
                    ].values

                    std = stats[
                        "std"
                    ].values

                    ax.plot(

                        x,
                        mean,

                        label=intensity,

                        color=intensity_colors[
                            intensity
                        ]
                    )

                    ax.fill_between(

                        x,

                        mean - std,

                        mean + std,

                        alpha=0.20,

                        color=intensity_colors[
                            intensity
                        ]
                    )

                # =============================================
                # STYLE
                # =============================================

                ax.axhline(

                    0,

                    linestyle="--",

                    linewidth=1,

                    color="black"
                )

                ax.set_title(

                    f"{model_name}\n"
                    f"{group_name}\n"
                    f"{title}"
                )

                ax.set_xlabel("Layer")

                if col_idx == 0:

                    ax.set_ylabel("Shift")

                ax.grid(True, alpha=0.3)

                ax.legend()

    plt.tight_layout()
    plt.show()

# =========================================================
# MAIN
# =========================================================

all_shift_df = {}

for model_name, cfg in MODELS.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    base_dir = cfg["base_dir"]

    probe_dir = cfg["probe_dir"]

    merged_csv = os.path.join(
        base_dir,
        "merged_results.csv"
    )

    # =====================================================
    # LOAD PROBES
    # =====================================================

    layers = get_all_probe_layers(
        probe_dir
    )

    probes = load_probes(
        probe_dir,
        layers
    )

    # =====================================================
    # RUN SHIFT ANALYSIS
    # =====================================================

    shift_df = run_shift_analysis_from_merged(

        merged_csv,
        probes
    )

    # =====================================================
    # SAVE
    # =====================================================

    save_path = os.path.join(

        base_dir,
        "shift_results.csv"
    )

    shift_df.to_csv(
        save_path,
        index=False
    )

    print(f"\nSaved:")
    print(save_path)

    all_shift_df[model_name] = shift_df

# =========================================================
# FINAL PLOT
# =========================================================

plot_shift_results(all_shift_df)

In [ ]:
import torch

def inspect_pt(path, verbose=False):

    data = torch.load(path, map_location="cpu")

    print(f"\n📦 File: {path}")

    # If wrapped dict (recommended format)
    if isinstance(data, dict) and "reps" in data:
        reps = data["reps"]
        print("Contains metadata:")
        for k in data:
            if k != "reps":
                print(f"  {k}: {data[k]}")
    else:
        reps = data

    print("\n🔑 Keys:")
    for k in reps.keys():
        print(f"  - {k}")

    print("\n📊 Tensor Info:")
    for k, v in reps.items():

        if not isinstance(v, torch.Tensor):
            print(f"{k}: ❌ Not a tensor ({type(v)})")
            continue

        print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}")

        if verbose:
            print(f"   mean={v.mean().item():.4f}, std={v.std().item():.4f}")

            # print small tensors fully
            if v.numel() < 20:
                print(f"   values={v}")

    print("\n✅ Done.\n")

In [ ]:
inspect_pt("/content/drive/MyDrive/probing/full_probe/albe/neutral_1001_IEO_ANG.pt")

In [ ]:
# =========================================================
# ACCURACY VISUALIZATION
# =========================================================
#
# TASK TYPES
#   style      -> speech cue task
#   intensity  -> intensity task
#
# STYLE TASK
#   binary classification:
#
#   neutral
#   emotional
#
#   any non-NEU emotion counts as emotional
#
#   direct model answer "emotional"
#   is VALID only for style task
#
# INTENSITY TASK
#   full emotion classification
#
# MD CONDITION
#   uses neutral_raw baseline
#
# =========================================================

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# CONFIG
# =========================================================

MODELS = {

    "QWEN": {
        "csv":
        "/content/drive/MyDrive/probing/full_probe/albe/merged_results.csv"
    },

    "FLAMINGO": {
        "csv":
        "/content/drive/MyDrive/probing/full_probe_flamingo/abla/merged_results.csv"
    }
}

# =========================================================
# EMOTION MAP
# =========================================================

EMOTION_MAP = {

    "ang": "ANG",
    "anger": "ANG",
    "angry": "ANG",

    "dis": "DIS",
    "disgust": "DIS",

    "fea": "FEA",
    "fear": "FEA",
    "fearful": "FEA",

    "hap": "HAP",
    "happy": "HAP",
    "happiness": "HAP",

    "sad": "SAD",
    "sadness": "SAD",

    "neu": "NEU",
    "neutral": "NEU",

    "sur": "SUR",
    "surprise": "SUR",
    "surprised": "SUR"
}

VALID_EMOTIONS = {
    "ANG",
    "DIS",
    "FEA",
    "HAP",
    "SAD",
    "NEU",
    "SUR"
}

STYLE_VALID = VALID_EMOTIONS.union({
    "EMOTIONAL"
})

# =========================================================
# HELPERS
# =========================================================

def extract_true_emotion(sample_id):

    """
    STYLE:
    1073_WSI_NEU_XX
              ↑
             NEU

    INTENSITY:
    1023_IEO_ANG
             ↑
            ANG
    """

    try:

        sample_id = str(sample_id)

        sample_id = os.path.splitext(sample_id)[0]

        parts = sample_id.split("_")

        # style task
        if parts[-1] == "XX":

            return parts[-2].upper()

        # intensity task
        else:

            return parts[-1].upper()

    except:

        return None


def extract_intensity(path):

    """
    Example:
    1088_IEO_SAD_MD.wav

    -> MD
    """

    try:

        filename = str(path).split("\\")[-1]

        filename = os.path.splitext(filename)[0]

        parts = filename.split("_")

        intensity = parts[-1].upper()

        if intensity in ["LO", "MD", "HI"]:
            return intensity

        return None

    except:

        return None


def normalize_prediction(text):

    """
    Normalize model prediction.
    """

    if pd.isna(text):
        return None

    text = str(text).lower().strip()

    # IMPORTANT:
    # direct binary answer for style task

    if re.search(r"\bemotional\b", text):
        return "EMOTIONAL"

    if re.search(r"\bneutral\b", text):
        return "NEU"

    for key, value in EMOTION_MAP.items():

        if re.search(rf"\b{key}\b", text):

            return value

    return None


def emotion_to_binary(label):

    """
    Convert to:
    emotional / neutral
    """

    if pd.isna(label):
        return None

    if label == "NEU":
        return "neutral"

    if label == "EMOTIONAL":
        return "emotional"

    return "emotional"

# =========================================================
# STORAGE
# =========================================================

speech_rows = []

intensity_rows = []

# =========================================================
# PROCESS MODELS
# =========================================================

for model_name, cfg in MODELS.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # =====================================================
    # LOAD
    # =====================================================

    df = pd.read_csv(cfg["csv"])

    print(f"Total rows: {len(df)}")

    # =====================================================
    # TRUE LABEL
    # =====================================================

    df["true_emotion"] = df[
        "id"
    ].apply(
        extract_true_emotion
    )

    # =====================================================
    # INTENSITY LEVEL
    # =====================================================

    df["intensity_level"] = df[
        "manipulated_path"
    ].apply(
        extract_intensity
    )

    # =====================================================
    # PREDICTIONS
    # =====================================================

    df["manip_pred"] = df[
        "raw_response"
    ].apply(
        normalize_prediction
    )

    df["baseline_pred"] = df[
        "neutral_raw"
    ].apply(
        normalize_prediction
    )

    # =====================================================
    # SPLIT TASKS
    # =====================================================

    speech_df = df[
        df["task"] == "style"
    ].copy()

    intensity_df = df[
        df["task"] == "intensity"
    ].copy()

    # =====================================================
    # TASK-SPECIFIC VALIDATION
    # =====================================================

    before = len(df)

    # -------------------------
    # STYLE TASK
    # -------------------------

    speech_df = speech_df[
        (
            speech_df["manip_pred"].isin(
                STYLE_VALID
            )
        )
        &
        (
            speech_df["baseline_pred"].isin(
                STYLE_VALID
            )
        )
    ].copy()

    # -------------------------
    # INTENSITY TASK
    # -------------------------

    intensity_df = intensity_df[
        (
            intensity_df["manip_pred"].isin(
                VALID_EMOTIONS
            )
        )
        &
        (
            intensity_df["baseline_pred"].isin(
                VALID_EMOTIONS
            )
        )
    ].copy()

    after = len(speech_df) + len(intensity_df)

    print(
        f"Malformed removed: "
        f"{before - after}"
    )
    # =====================================================
    # STYLE LABEL DISTRIBUTION
    # =====================================================

    speech_df["true_binary"] = speech_df[
        "true_emotion"
    ].apply(
        emotion_to_binary
    )

    speech_df["manip_binary"] = speech_df[
        "manip_pred"
    ].apply(
        emotion_to_binary
    )

    speech_df["baseline_binary"] = speech_df[
        "baseline_pred"
    ].apply(
        emotion_to_binary
    )

    print("\nSTYLE TASK LABEL DISTRIBUTION")

    # -----------------------------------------------------
    # TRUE
    # -----------------------------------------------------

    true_dist = (
        speech_df["true_binary"]
        .value_counts(normalize=True)
        * 100
    )

    print("\nTrue distribution (%):")
    print(
        true_dist.round(2)
    )

    # -----------------------------------------------------
    # BASELINE
    # -----------------------------------------------------

    baseline_dist = (
        speech_df["baseline_binary"]
        .value_counts(normalize=True)
        * 100
    )

    print("\nBaseline prediction distribution (%):")
    print(
        baseline_dist.round(2)
    )

    # -----------------------------------------------------
    # MANIPULATED
    # -----------------------------------------------------

    manip_dist = (
        speech_df["manip_binary"]
        .value_counts(normalize=True)
        * 100
    )

    print("\nSpeech cue prediction distribution (%):")
    print(
        manip_dist.round(2)
    )

    # =====================================================
    # STYLE TASK
    # BINARY CLASSIFICATION
    # =====================================================

    speech_df["true_binary"] = speech_df[
        "true_emotion"
    ].apply(
        emotion_to_binary
    )

    speech_df["manip_binary"] = speech_df[
        "manip_pred"
    ].apply(
        emotion_to_binary
    )

    speech_df["baseline_binary"] = speech_df[
        "baseline_pred"
    ].apply(
        emotion_to_binary
    )

    speech_df["manip_correct"] = (
        speech_df["manip_binary"]
        ==
        speech_df["true_binary"]
    ).astype(int)

    speech_df["baseline_correct"] = (
        speech_df["baseline_binary"]
        ==
        speech_df["true_binary"]
    ).astype(int)

    # =====================================================
    # INTENSITY TASK
    # FULL EMOTION MATCH
    # =====================================================

    intensity_df["manip_correct"] = (
        intensity_df["manip_pred"]
        ==
        intensity_df["true_emotion"]
    ).astype(int)

    intensity_df["baseline_correct"] = (
        intensity_df["baseline_pred"]
        ==
        intensity_df["true_emotion"]
    ).astype(int)

    # =====================================================
    # OVERALL ACCURACY
    # =====================================================

    overall_baseline = pd.concat([
        speech_df["baseline_correct"],
        intensity_df["baseline_correct"]
    ]).mean()

    overall_manip = pd.concat([
        speech_df["manip_correct"],
        intensity_df["manip_correct"]
    ]).mean()

    print(
        f"Baseline accuracy: "
        f"{overall_baseline:.4f}"
    )

    print(
        f"Manipulated accuracy: "
        f"{overall_manip:.4f}"
    )

    # =====================================================
    # SPEECH TASK RESULTS
    # =====================================================

    print(
        f"Speech rows: "
        f"{len(speech_df)}"
    )

    speech_rows.append({

        "model": model_name,
        "condition": "Baseline",
        "accuracy": speech_df[
            "baseline_correct"
        ].mean()
    })

    speech_rows.append({

        "model": model_name,
        "condition": "Speech Cue",
        "accuracy": speech_df[
            "manip_correct"
        ].mean()
    })

    # =====================================================
    # INTENSITY TASK RESULTS
    # =====================================================

    print(
        f"Intensity rows: "
        f"{len(intensity_df)}"
    )

    print(
        intensity_df[
            "intensity_level"
        ].value_counts(dropna=False)
    )

    for intensity in ["LO", "MD", "HI"]:

        # =================================================
        # MD = BASELINE (neutral_raw)
        # =================================================

        if intensity == "MD":

            subset = intensity_df.copy()

            accuracy = subset[
                "baseline_correct"
            ].mean()

        # =================================================
        # LO / HI = manipulated audio
        # =================================================

        else:

            subset = intensity_df[
                intensity_df["intensity_level"]
                == intensity
            ]

            accuracy = subset[
                "manip_correct"
            ].mean()

        print(
            f"{model_name} {intensity}: "
            f"{len(subset)} rows"
        )

        intensity_rows.append({

            "model": model_name,
            "intensity": intensity,
            "accuracy": accuracy
        })

# =========================================================
# DATAFRAMES
# =========================================================

speech_plot_df = pd.DataFrame(
    speech_rows
)

intensity_plot_df = pd.DataFrame(
    intensity_rows
)

print("\nSpeech plot dataframe:")
print(speech_plot_df)

print("\nIntensity plot dataframe:")
print(intensity_plot_df)

# =========================================================
# VISUALIZATION
# =========================================================

sns.set_style("whitegrid")

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

# =========================================================
# SPEECH TASK
# =========================================================

sns.barplot(
    data=speech_plot_df,
    x="condition",
    y="accuracy",
    hue="model",
    ax=axes[0]
)

axes[0].set_title(
    "Speech Cue Task Accuracy"
)

axes[0].set_ylabel(
    "Accuracy"
)

axes[0].set_ylim(0, 1)

# =========================================================
# INTENSITY TASK
# =========================================================

sns.barplot(
    data=intensity_plot_df,
    x="intensity",
    y="accuracy",
    hue="model",
    order=["LO", "MD", "HI"],
    ax=axes[1]
)

axes[1].set_title(
    "Intensity Task Accuracy"
)

axes[1].set_ylabel(
    "Accuracy"
)

axes[1].set_ylim(0, 1)

# =========================================================
# FINALIZE
# =========================================================

plt.tight_layout()

plt.show()

# =========================================================
# FINAL TABLES
# =========================================================

print("\n" + "=" * 60)
print("SPEECH TASK RESULTS")
print("=" * 60)

print(speech_plot_df)

print("\n" + "=" * 60)
print("INTENSITY TASK RESULTS")
print("=" * 60)

print(intensity_plot_df)

In [ ]:
# =========================================================
# INTENSITY ACCURACY VISUALIZATION ONLY
# =========================================================

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# CONFIG
# =========================================================

MODELS = {

    "QWEN": {
        "csv":
        "/content/drive/MyDrive/probing/full_probe/albe/merged_results.csv"
    },

    "FLAMINGO": {
        "csv":
        "/content/drive/MyDrive/probing/full_probe_flamingo/abla/merged_results.csv"
    }
}

# =========================================================
# EMOTION MAP
# =========================================================

EMOTION_MAP = {

    "ang": "ANG",
    "anger": "ANG",
    "angry": "ANG",

    "dis": "DIS",
    "disgust": "DIS",

    "fea": "FEA",
    "fear": "FEA",
    "fearful": "FEA",

    "hap": "HAP",
    "happy": "HAP",
    "happiness": "HAP",

    "sad": "SAD",
    "sadness": "SAD",

    "neu": "NEU",
    "neutral": "NEU",

    "sur": "SUR",
    "surprise": "SUR",
    "surprised": "SUR"
}

VALID_EMOTIONS = {
    "ANG",
    "DIS",
    "FEA",
    "HAP",
    "SAD",
    "NEU",
    "SUR"
}

# =========================================================
# HELPERS
# =========================================================

def extract_true_emotion(sample_id):

    try:

        sample_id = str(sample_id)

        sample_id = os.path.splitext(sample_id)[0]

        parts = sample_id.split("_")

        return parts[-1].upper()

    except:

        return None


def extract_intensity(path):

    try:

        filename = str(path).split("\\")[-1]

        filename = os.path.splitext(filename)[0]

        parts = filename.split("_")

        intensity = parts[-1].upper()

        if intensity in ["LO", "MD", "HI"]:
            return intensity

        return None

    except:

        return None


def normalize_prediction(text):

    if pd.isna(text):
        return None

    text = str(text).lower().strip()

    if re.search(r"\bneutral\b", text):
        return "NEU"

    for key, value in EMOTION_MAP.items():

        if re.search(rf"\b{key}\b", text):

            return value

    return None

# =========================================================
# STORAGE
# =========================================================

intensity_rows = []

# =========================================================
# PROCESS MODELS
# =========================================================

for model_name, cfg in MODELS.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    df = pd.read_csv(cfg["csv"])

    # -----------------------------------------------------
    # intensity task only
    # -----------------------------------------------------

    df = df[
        df["task"] == "intensity"
    ].copy()

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    df["true_emotion"] = df[
        "id"
    ].apply(
        extract_true_emotion
    )

    df["intensity_level"] = df[
        "manipulated_path"
    ].apply(
        extract_intensity
    )

    df["manip_pred"] = df[
        "raw_response"
    ].apply(
        normalize_prediction
    )

    df["baseline_pred"] = df[
        "neutral_raw"
    ].apply(
        normalize_prediction
    )

    # -----------------------------------------------------
    # valid rows
    # -----------------------------------------------------

    df = df[
        (
            df["manip_pred"].isin(
                VALID_EMOTIONS
            )
        )
        &
        (
            df["baseline_pred"].isin(
                VALID_EMOTIONS
            )
        )
    ].copy()

    print(f"Valid rows: {len(df)}")

    # -----------------------------------------------------
    # correctness
    # -----------------------------------------------------

    df["manip_correct"] = (
        df["manip_pred"]
        ==
        df["true_emotion"]
    ).astype(int)

    df["baseline_correct"] = (
        df["baseline_pred"]
        ==
        df["true_emotion"]
    ).astype(int)

    # =====================================================
    # INTENSITY RESULTS
    # =====================================================

    for intensity in ["LO", "MD", "HI"]:

        # MD uses neutral baseline

        if intensity == "MD":

            subset = df.copy()

            accuracy = subset[
                "baseline_correct"
            ].mean()

        else:

            subset = df[
                df["intensity_level"]
                == intensity
            ]

            accuracy = subset[
                "manip_correct"
            ].mean()

        print(
            f"{model_name} {intensity}: "
            f"{accuracy:.4f}"
        )

        intensity_rows.append({

            "model": model_name,
            "intensity": intensity,
            "accuracy": accuracy
        })

# =========================================================
# DATAFRAME
# =========================================================

intensity_plot_df = pd.DataFrame(
    intensity_rows
)

print("\nIntensity plot dataframe:")
print(intensity_plot_df)

# =========================================================
# PLOT
# =========================================================

sns.set_style("whitegrid")

plt.figure(figsize=(7, 5))

sns.barplot(
    data=intensity_plot_df,
    x="intensity",
    y="accuracy",
    hue="model",
    order=["LO", "MD", "HI"]
)

plt.title(
    "Intensity Task Accuracy"
)

plt.ylabel(
    "Accuracy"
)

plt.xlabel(
    "Intensity"
)

plt.ylim(0, 1)

plt.tight_layout()

plt.show()

In [ ]:
# =========================================================
# DEMOGRAPHIC ANALYSIS
# REMOVE UNKNOWN GROUPS
# 2 x 4 PANEL
# =========================================================

import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# CONFIG
# =========================================================

MODEL_FILES = {

    "QWEN":
    "/content/drive/MyDrive/probe/probe_representations_2/probe_rep_index.csv",

    "FLAMINGO":
    "/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_rep_index.csv"
}

DEMOGRAPHIC_CSV = (
    "/content/drive/MyDrive/probing/VideoDemographics.csv"
)

# =========================================================
# LOAD DEMOGRAPHICS
# =========================================================

demo_df = pd.read_csv(
    DEMOGRAPHIC_CSV
)

demo_df["ActorID"] = (
    demo_df["ActorID"]
    .astype(str)
)

# =========================================================
# EMOTION MAP
# =========================================================

EMOTION_MAP = {

    "ang": "ANG",
    "anger": "ANG",
    "angry": "ANG",

    "dis": "DIS",
    "disgust": "DIS",
    "disgusted": "DIS",

    "fea": "FEA",
    "fear": "FEA",
    "fearful": "FEA",

    "hap": "HAP",
    "happy": "HAP",

    "sad": "SAD",
    "sadness": "SAD",

    "neu": "NEU",
    "neutral": "NEU",

    "sur": "SUR",
    "surprise": "SUR",
    "surprised": "SUR"
}

VALID_EMOTIONS = {
    "ANG",
    "DIS",
    "FEA",
    "HAP",
    "SAD",
    "NEU",
    "SUR"
}

# =========================================================
# HELPERS
# =========================================================

def normalize_prediction(text):

    if pd.isna(text):
        return None

    text = str(text).lower().strip()

    for key, value in EMOTION_MAP.items():

        if re.search(rf"\b{key}\b", text):

            return value

    return None


def age_bucket(age):

    try:

        age = int(age)

        if age < 35:
            return "Young"

        elif age < 50:
            return "Middle"

        else:
            return "Older"

    except:

        return None

# =========================================================
# STORAGE
# =========================================================

all_results = []
all_distribution_rows = []

# =========================================================
# PROCESS
# =========================================================

for model_name, csv_path in MODEL_FILES.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # =====================================================
    # LOAD
    # =====================================================

    df = pd.read_csv(csv_path)

    print(f"Rows: {len(df)}")

    # =====================================================
    # TRUE LABEL
    # =====================================================

    df["true_emotion"] = (
        df["emotion"]
        .astype(str)
        .str.upper()
    )

    # =====================================================
    # MODEL OUTPUT
    # =====================================================

    df["pred_emotion"] = df[
        "llm_output"
    ].apply(
        normalize_prediction
    )

    # =====================================================
    # FILTER VALID
    # =====================================================

    before = len(df)

    df = df[
        df["pred_emotion"].isin(
            VALID_EMOTIONS
        )
    ].copy()

    after = len(df)

    print(
        f"Malformed removed: "
        f"{before - after}"
    )

    # =====================================================
    # ACCURACY
    # =====================================================

    df["correct"] = (
        df["pred_emotion"]
        ==
        df["true_emotion"]
    ).astype(int)

    # =====================================================
    # ACTOR ID
    # =====================================================

    df["ActorID"] = (
        df["actor"]
        .astype(str)
    )

    # =====================================================
    # MERGE DEMOGRAPHICS
    # =====================================================

    df = df.merge(
        demo_df,
        on="ActorID",
        how="left"
    )

    # =====================================================
    # REMOVE UNKNOWN RACE
    # =====================================================

    df = df[
        df["Race"].notna()
    ]

    df = df[
        df["Race"] != "Unknown"
    ]

    # =====================================================
    # AGE GROUP
    # =====================================================

    df["AgeGroup"] = df[
        "Age"
    ].apply(
        age_bucket
    )

    # =====================================================
    # DISTRIBUTION
    # =====================================================

    for demographic in [
        "Sex",
        "Race",
        "Ethnicity",
        "AgeGroup"
    ]:

        dist = (
            df[demographic]
            .value_counts(dropna=False)
            .reset_index()
        )

        dist.columns = [
            demographic,
            "count"
        ]

        dist["percentage"] = (
            dist["count"]
            / dist["count"].sum()
            * 100
        )

        dist["model"] = model_name

        dist["demographic_type"] = demographic

        all_distribution_rows.append(
            dist
        )

    # =====================================================
    # ACCURACY
    # =====================================================

    for demographic in [
        "Sex",
        "Race",
        "Ethnicity",
        "AgeGroup"
    ]:

        stats = (
            df.groupby(
                demographic
            )["correct"]
            .agg(
                ["mean", "count"]
            )
            .reset_index()
        )

        stats.rename(
            columns={
                "mean": "accuracy",
                "count": "samples"
            },
            inplace=True
        )

        stats["model"] = model_name

        stats["demographic_type"] = demographic

        all_results.append(
            stats
        )

# =========================================================
# FINAL DATAFRAMES
# =========================================================

results_df = pd.concat(
    all_results,
    ignore_index=True
)

distribution_df = pd.concat(
    all_distribution_rows,
    ignore_index=True
)

# =========================================================
# STYLE
# =========================================================

sns.set_style("whitegrid")

demographics = [
    "Sex",
    "Race",
    "Ethnicity",
    "AgeGroup"
]

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(
    2,
    4,
    figsize=(24, 10)
)

# =========================================================
# TOP ROW: DISTRIBUTION
# =========================================================

for col_idx, demographic in enumerate(demographics):

    ax = axes[0, col_idx]

    subset = distribution_df[
        distribution_df["demographic_type"]
        == demographic
    ]

    sns.barplot(
        data=subset,
        x=demographic,
        y="count",
        hue="model",
        ax=ax
    )

    ax.set_title(
        f"{demographic} Distribution"
    )

    ax.set_ylabel("Count")

    ax.tick_params(
        axis="x",
        rotation=20
    )

# =========================================================
# BOTTOM ROW: ACCURACY
# =========================================================

for col_idx, demographic in enumerate(demographics):

    ax = axes[1, col_idx]

    subset = results_df[
        results_df["demographic_type"]
        == demographic
    ]

    sns.barplot(
        data=subset,
        x=demographic,
        y="accuracy",
        hue="model",
        ax=ax
    )

    ax.set_title(
        f"{demographic} Accuracy"
    )

    ax.set_ylabel("Accuracy")

    ax.set_ylim(0, 1)

    ax.tick_params(
        axis="x",
        rotation=20
    )

# =========================================================
# REMOVE EXTRA LEGENDS
# =========================================================

for i in range(2):
    for j in range(4):

        if j != 3:
            axes[i, j].legend_.remove()

# =========================================================
# FINALIZE
# =========================================================

plt.tight_layout()

plt.show()

# =========================================================
# SAVE
# =========================================================

results_df.to_csv(
    "demographic_accuracy_results.csv",
    index=False
)

distribution_df.to_csv(
    "demographic_distribution.csv",
    index=False
)

print("\nSaved:")
print("demographic_accuracy_results.csv")
print("demographic_distribution.csv")

In [ ]:
# ============================================================
# REPRESENTATION BASELINE DEVIATION
# ============================================================

import os
import torch
import pandas as pd
import numpy as np

from glob import glob
from tqdm import tqdm
from torch.nn.functional import cosine_similarity

# ============================================================
# CONFIG
# ============================================================

RUN1_DIR = "/content/drive/MyDrive/probing/full_probe"
RUN2_DIR = "/content/drive/MyDrive/probing/full_probe/albe"

SAVE_PATH = (
    "/content/drive/MyDrive/probing/"
    "qwen_baseline_deviation.csv"
)

# ============================================================
# HELPERS
# ============================================================

def cosine_distance(x, y):

    x = x.flatten().float()
    y = y.flatten().float()

    sim = cosine_similarity(
        x.unsqueeze(0),
        y.unsqueeze(0)
    ).item()

    return 1.0 - sim

# ============================================================
# LOAD FILES
# ============================================================

run1_files = {
    os.path.basename(f): f
    for f in glob(os.path.join(RUN1_DIR, "*.pt"))
}

run2_files = {
    os.path.basename(f): f
    for f in glob(os.path.join(RUN2_DIR, "*.pt"))
}

shared_files = sorted(
    set(run1_files.keys())
    & set(run2_files.keys())
)

print(f"Matched files: {len(shared_files)}")

# ============================================================
# MAIN LOOP
# ============================================================

rows = []

for fname in tqdm(shared_files):

    path1 = run1_files[fname]
    path2 = run2_files[fname]

    rep1 = torch.load(path1, map_location="cpu")
    rep2 = torch.load(path2, map_location="cpu")

    shared_layers = (
        set(rep1.keys())
        & set(rep2.keys())
    )

    for layer in shared_layers:

        try:

            d = cosine_distance(
                rep1[layer],
                rep2[layer]
            )

            rows.append({

                "file": fname,
                "layer": layer,
                "cosine_distance": d
            })

        except Exception as e:

            print(
                f"Error: {fname} | {layer} | {e}"
            )

# ============================================================
# SAVE
# ============================================================

df = pd.DataFrame(rows)

df.to_csv(SAVE_PATH, index=False)

print(f"\nSaved: {SAVE_PATH}")

# ============================================================
# SUMMARY
# ============================================================

summary = (
    df.groupby("layer")["cosine_distance"]
    .agg(["mean", "std", "count"])
)

print(summary)

In [ ]:
# ============================================================
# REPRESENTATION BASELINE DEVIATION
# ============================================================

import os
import torch
import pandas as pd
import numpy as np

from glob import glob
from tqdm import tqdm
from torch.nn.functional import cosine_similarity

# ============================================================
# CONFIG
# ============================================================

RUN1_DIR = "/content/drive/MyDrive/probing/full_probe_flamingo"
RUN2_DIR = "/content/drive/MyDrive/probing/full_probe_flamingo/abla"

SAVE_PATH = (
    "/content/drive/MyDrive/probing/"
    "flamingo_baseline_deviation.csv"
)

# ============================================================
# HELPERS
# ============================================================

def cosine_distance(x, y):

    x = x.flatten().float()
    y = y.flatten().float()

    sim = cosine_similarity(
        x.unsqueeze(0),
        y.unsqueeze(0)
    ).item()

    return 1.0 - sim

# ============================================================
# LOAD FILES
# ============================================================

run1_files = {
    os.path.basename(f): f
    for f in glob(os.path.join(RUN1_DIR, "*.pt"))
}

run2_files = {
    os.path.basename(f): f
    for f in glob(os.path.join(RUN2_DIR, "*.pt"))
}

shared_files = sorted(
    set(run1_files.keys())
    & set(run2_files.keys())
)

print(f"Matched files: {len(shared_files)}")

# ============================================================
# MAIN LOOP
# ============================================================

rows = []

for fname in tqdm(shared_files):

    path1 = run1_files[fname]
    path2 = run2_files[fname]

    rep1 = torch.load(path1, map_location="cpu")
    rep2 = torch.load(path2, map_location="cpu")

    shared_layers = (
        set(rep1.keys())
        & set(rep2.keys())
    )

    for layer in shared_layers:

        try:

            d = cosine_distance(
                rep1[layer],
                rep2[layer]
            )

            rows.append({

                "file": fname,
                "layer": layer,
                "cosine_distance": d
            })

        except Exception as e:

            print(
                f"Error: {fname} | {layer} | {e}"
            )

# ============================================================
# SAVE
# ============================================================

df = pd.DataFrame(rows)

df.to_csv(SAVE_PATH, index=False)

print(f"\nSaved: {SAVE_PATH}")

# ============================================================
# SUMMARY
# ============================================================

summary = (
    df.groupby("layer")["cosine_distance"]
    .agg(["mean", "std", "count"])
)

print(summary)

In [ ]:
# =========================================================
# PURE COSINE REPRESENTATION SHIFT ANALYSIS
# =========================================================

import os
import re
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch.nn.functional import cosine_similarity

# =========================================================
# CONFIG
# =========================================================

MODELS = {

    "QWEN": {
        "base_dir":
            "/content/drive/MyDrive/probing/full_probe/albe",
    },

    "FLAMINGO": {
        "base_dir":
            "/content/drive/MyDrive/probing/full_probe_flamingo/abla",
    }
}

# =========================================================
# HELPERS
# =========================================================

def natural_sort_key(layer):

    nums = re.findall(r"\d+", layer)

    if nums:
        return (
            layer.split(nums[-1])[0],
            int(nums[-1])
        )

    return (layer, 0)

# =========================================================
# REPRESENTATION PREPROCESS
# =========================================================

def preprocess_rep(x, layer):

    x = x.float()

    # =====================================================
    # LM pooling
    # =====================================================

    if layer.startswith("lm_") and x.dim() == 3:
        x = x.mean(dim=1)

    # =====================================================
    # Shape handling
    # =====================================================

    if x.dim() == 2 and x.shape[0] == 1:
        x = x.squeeze(0)

    elif x.dim() == 2:
        x = x.mean(dim=0)

    elif x.dim() > 2:
        x = x.flatten()

    return x.flatten()

# =========================================================
# COSINE DISTANCE
# =========================================================

def cosine_distance(x, y):

    sim = cosine_similarity(
        x.unsqueeze(0),
        y.unsqueeze(0)
    ).item()

    return 1.0 - sim

# =========================================================
# SHIFT ANALYSIS
# =========================================================

def run_cosine_shift_analysis(merged_csv):

    df = pd.read_csv(merged_csv)

    print("\nDetected task types:")
    print(df["task"].value_counts())

    all_results = []

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):

        manip_rep_path = row["rep_path"]
        neutral_rep_path = row["neutral_rep_path"]

        # =================================================
        # LOAD REPRESENTATIONS
        # =================================================

        try:

            manip_rep = torch.load(
                manip_rep_path,
                map_location="cpu"
            )

            neutral_rep = torch.load(
                neutral_rep_path,
                map_location="cpu"
            )

        except Exception as e:

            print(f"Representation load error: {e}")
            continue

        # =================================================
        # SHARED LAYERS
        # =================================================

        shared_layers = (
            set(manip_rep.keys())
            & set(neutral_rep.keys())
        )

        # =================================================
        # LAYER LOOP
        # =================================================

        for layer in shared_layers:

            try:

                x_manip = preprocess_rep(
                    manip_rep[layer],
                    layer
                )

                x_neutral = preprocess_rep(
                    neutral_rep[layer],
                    layer
                )

                # ============================================
                # COSINE DISTANCE
                # ============================================

                cos_dist = cosine_distance(
                    x_manip,
                    x_neutral
                )

                # ============================================
                # SAVE
                # ============================================

                all_results.append({

                    "id": row["id"],
                    "task": row["task"],
                    "layer": layer,

                    "cosine_distance": cos_dist,

                    "condition":
                        row.get("condition", None),

                    "baseline_type":
                        row.get("baseline_type", None),

                    "intensity":
                        row.get("intensity", None),
                })

            except Exception as e:

                print(f"Error: {layer} | {e}")
                continue

    return pd.DataFrame(all_results)

# =========================================================
# VISUALIZATION
# =========================================================

def plot_cosine_results(all_shift_df):

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(18, 10)
    )

    for row_idx, (
        model_name,
        shift_df
    ) in enumerate(all_shift_df.items()):

        # =================================================
        # AUDIO / LLM SPLIT
        # =================================================

        audio_df = shift_df[
            ~shift_df["layer"].str.startswith("lm_")
        ]

        llm_df = shift_df[
            shift_df["layer"].str.startswith("lm_")
        ]

        groups = [
            ("Audio + Projector", audio_df),
            ("LLM", llm_df)
        ]

        # =================================================
        # PLOT
        # =================================================

        for col_idx, (
            group_name,
            group_df
        ) in enumerate(groups):

            ax = axes[row_idx, col_idx]

            task_types = sorted(
                group_df["task"].unique()
            )

            colors = ["gold", "blue"]

            for task_i, task_name in enumerate(task_types):

                subset = group_df[
                    group_df["task"] == task_name
                ]

                stats = subset.groupby(
                    "layer"
                )["cosine_distance"].agg(
                    ["mean", "std"]
                ).reset_index()

                stats["sort_key"] = stats[
                    "layer"
                ].apply(natural_sort_key)

                stats = stats.sort_values(
                    "sort_key"
                )

                x = np.arange(len(stats))

                mean = stats["mean"].values
                std = stats["std"].values

                color = colors[
                    task_i % len(colors)
                ]

                ax.plot(
                    x,
                    mean,
                    label=task_name,
                )

                ax.fill_between(
                    x,
                    mean - std,
                    mean + std,
                    alpha=0.20
                )

            # =============================================
            # STYLE
            # =============================================

            ax.set_title(
                f"{model_name}\n"
                f"{group_name}\n"
                f"Cosine Distance"
            )

            ax.set_xlabel("Layer")
            ax.set_ylabel("Cosine Distance")

            ax.grid(True, alpha=0.3)

            ax.legend()

    plt.tight_layout()
    plt.show()

# =========================================================
# MAIN
# =========================================================

all_shift_df = {}

for model_name, cfg in MODELS.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    base_dir = cfg["base_dir"]

    merged_csv = os.path.join(
        base_dir,
        "merged_results.csv"
    )

    # =====================================================
    # RUN ANALYSIS
    # =====================================================

    shift_df = run_cosine_shift_analysis(
        merged_csv
    )

    # =====================================================
    # SAVE
    # =====================================================

    save_path = os.path.join(
        base_dir,
        "cosine_shift_results.csv"
    )

    shift_df.to_csv(
        save_path,
        index=False
    )

    print(f"\nSaved:")
    print(save_path)

    # =====================================================
    # SUMMARY
    # =====================================================

    summary = (
        shift_df.groupby("layer")[
            "cosine_distance"
        ]
        .agg(["mean", "std", "count"])
    )

    print("\nLayer Summary:")
    print(summary)

    all_shift_df[model_name] = shift_df

# =========================================================
# FINAL PLOT
# =========================================================

plot_cosine_results(all_shift_df)